In [1]:
# ==============================================================================
# CELDA 1: LIBRERÍAS + INSPECCIÓN DE ARCHIVOS IVE (2018-19, ambas hojas)
# ==============================================================================
import pandas as pd
import openpyxl
from google.colab import drive
drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/IVE/'

# Empezamos con el que parece más directo para tu bienio 2018-19
archivo_2018 = 'IVE-POR-RBD-BASICA-MEDIA-COMUNA-2018.xlsx'

wb = openpyxl.load_workbook(RUTA + archivo_2018, read_only=True, data_only=True)
print("Hojas:", wb.sheetnames)

for hoja in wb.sheetnames:
    df_preview = pd.read_excel(RUTA + archivo_2018, sheet_name=hoja, nrows=5)
    print(f"\n--- Hoja: {hoja} ---")
    print(df_preview.columns.tolist())
    print(df_preview.head(3))

Mounted at /content/drive
Hojas: ['BASICA', 'MEDIA', 'COMUNA']

--- Hoja: BASICA ---
['RBD', 'DV_RBD', 'NOMBRE_ESTABLECIMIENTO', 'DEPENDENCIA', 'AREA', 'COD_REGION', 'COD_PROV', 'COD_COM_EST', 'COMUNA', 'PRIMERA PRIORIDAD', 'SEGUNDA PRIORIDAD', 'TERCERA PRIORIDAD', 'NO VULNERABLES', 'SIN INFORMACION', 'TOTAL MATRICULA BASICA 2017', 'IVE-SINAE BASICA 2018']
   RBD  DV_RBD                   NOMBRE_ESTABLECIMIENTO     DEPENDENCIA  \
0    5       1                 JOVINA NARANJO FERNANDEZ  Municipal DAEM   
1    8       6  COLEGIO INTEGRADO EDUARDO FREI MONTALVA  Municipal DAEM   
2    9       4              ESCUELA REPUBLICA DE ISRAEL  Municipal DAEM   

     AREA  COD_REGION  COD_PROV  COD_COM_EST COMUNA  PRIMERA PRIORIDAD  \
0  Urbano          15       151        15101  Arica                148   
1  Urbano          15       151        15101  Arica                185   
2  Urbano          15       151        15101  Arica                797   

   SEGUNDA PRIORIDAD  TERCERA PRIORIDAD  NO

In [2]:
# ==============================================================================
# CELDA 2: INSPECCIÓN DEL ARCHIVO 2019
# ==============================================================================
archivo_2019 = 'IVE-2019-1.xlsx'
wb2 = openpyxl.load_workbook(RUTA + archivo_2019, read_only=True, data_only=True)
print("Hojas:", wb2.sheetnames)

for hoja in wb2.sheetnames:
    df_preview = pd.read_excel(RUTA + archivo_2019, sheet_name=hoja, nrows=3)
    print(f"\n--- Hoja: {hoja} ---")
    print(df_preview.columns.tolist())

Hojas: ['BASICA', 'MEDIA', 'COMUNA']

--- Hoja: BASICA ---
['ID_RBD', 'DV_RBD', 'DS_NOM_ESTABLE', 'DS_TIPO_DEPENDENCIA', 'DS_RURALIDAD', 'ID_REGION_ESTABLE', 'ID_PROVINCIA_ESTABLE', 'ID_COMUNA_ESTABLE', 'DS_COMUNA_ESTABLE', 'PRIMERA PRIORIDAD', 'SEGUNDA PRIORIDAD', 'TERCERA PRIORIDAD', 'NO PRIORIZADO EN VULNERABILIDAD', 'SIN INFORMACION', 'TOTAL MATRICULA BASICA 2018', 'IVE-SINAE BASICA 2019']

--- Hoja: MEDIA ---
['ID_RBD', 'DV_RBD', 'DS_NOM_ESTABLE', 'DS_TIPO_DEPENDENCIA', 'DS_RURALIDAD', 'ID_REGION_ESTABLE', 'ID_PROVINCIA_ESTABLE', 'ID_COMUNA_ESTABLE', 'DS_COMUNA_ESTABLE', 'PRIMERA PRIORIDAD', 'SEGUNDA PRIORIDAD', 'TERCERA PRIORIDAD', 'NO PRIORIZADO EN VULNERABILIDAD', 'SIN INFORMACION', 'TOTAL MATRICULA MEDIA 2018', 'IVE-SINAE MEDIA 2019']

--- Hoja: COMUNA ---
['ID COMUNA', 'COMUNA', 'PRIMERA PRIORIDAD', 'SEGUNDA PRIORIDAD', 'TERCERA PRIORIDAD', 'NO PRIORIZADO EN VULNERABILIDAD', 'SIN INFORMACION', 'TOTAL MATRICULA BASICA-MEDIA 2019', 'IVE-SINAE COMUNAL 2019']


In [3]:
# ==============================================================================
# CELDA 3: INGESTA Y ESTANDARIZACIÓN IVE 2018-19 (ambas hojas, ambos esquemas)
# ==============================================================================
def procesar_ive(archivo, anio, hoja, nivel):
    df = pd.read_excel(RUTA + archivo, sheet_name=hoja)
    d = df.copy()
    d.columns = [c.strip().upper() for c in d.columns]

    # Normalización de nombres según esquema (2018 vs 2019+)
    col_rbd = 'RBD' if 'RBD' in d.columns else 'ID_RBD'
    col_nombre = 'NOMBRE_ESTABLECIMIENTO' if 'NOMBRE_ESTABLECIMIENTO' in d.columns else 'DS_NOM_ESTABLE'
    col_ive = [c for c in d.columns if c.startswith('IVE-SINAE')][0]

    d['rbd'] = pd.to_numeric(d[col_rbd], errors='coerce')
    d = d.dropna(subset=['rbd'])
    d['rbd'] = d['rbd'].astype('Int64').astype(str)
    d['nom_rbd_ive'] = d[col_nombre].astype(str).str.strip().str.upper()
    d[f'ive_{nivel}'] = pd.to_numeric(d[col_ive], errors='coerce')

    return d[['rbd', 'nom_rbd_ive', f'ive_{nivel}']]

archivos_ive = {
    2018: 'IVE-POR-RBD-BASICA-MEDIA-COMUNA-2018.xlsx',
    2019: 'IVE-2019-1.xlsx',
}

ive_basica = {}
ive_media = {}
for anio, archivo in archivos_ive.items():
    ive_basica[anio] = procesar_ive(archivo, anio, 'BASICA', 'basica')
    ive_media[anio] = procesar_ive(archivo, anio, 'MEDIA', 'media')
    print(f"{anio} | Básica: {len(ive_basica[anio])} colegios | Media: {len(ive_media[anio])} colegios")

RUTA_SALIDA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
import pickle
with open(RUTA_SALIDA + 'ive_temp.pkl', 'wb') as f:
    pickle.dump({'basica': ive_basica, 'media': ive_media}, f)
print("Guardado temporal OK")

2018 | Básica: 7823 colegios | Media: 2577 colegios
2019 | Básica: 7695 colegios | Media: 2551 colegios
Guardado temporal OK


In [4]:
# ==============================================================================
# CELDA 4: VERIFICACIÓN CRUZADA DE NOMBRES (IVE vs SIMCE) POR RBD
# ==============================================================================
import re

def normalizar_nombre(s):
    s = str(s).upper().strip()
    s = re.sub(r'[^A-Z0-9 ]', '', s)  # quita puntuación/tildes raras
    s = re.sub(r'\s+', ' ', s)
    return s

df_simce = pd.read_parquet(RUTA_SALIDA + 'simce_maestro_bienios.parquet')
nombres_simce = df_simce[['rbd', 'nom_rbd']].drop_duplicates(subset='rbd').copy()
nombres_simce['nom_norm_simce'] = nombres_simce['nom_rbd'].apply(normalizar_nombre)

resultados = []
for anio in [2018, 2019]:
    for nivel, ive_dict in [('basica', ive_basica), ('media', ive_media)]:
        d = ive_dict[anio].copy()
        d['nom_norm_ive'] = d['nom_rbd_ive'].apply(normalizar_nombre)

        cruce = pd.merge(d, nombres_simce, on='rbd', how='inner')
        cruce['coincide'] = cruce['nom_norm_ive'] == cruce['nom_norm_simce']

        n_total = len(cruce)
        n_coincide = cruce['coincide'].sum()
        pct = n_coincide / n_total * 100 if n_total > 0 else 0

        resultados.append({'anio': anio, 'nivel': nivel, 'n_cruzados': n_total,
                            'coincidencia_exacta': n_coincide, 'pct_coincidencia': round(pct, 1)})

        # Muestra de discrepancias (para inspección)
        discrepancias = cruce[~cruce['coincide']][['rbd', 'nom_rbd_ive', 'nom_rbd']].head(5)
        if len(discrepancias) > 0:
            print(f"\n--- Discrepancias {anio}/{nivel} (muestra) ---")
            print(discrepancias.to_string(index=False))

print("\n--- Resumen de verificación cruzada ---")
print(pd.DataFrame(resultados).to_string(index=False))


--- Discrepancias 2018/basica (muestra) ---
rbd                            nom_rbd_ive                             nom_rbd
280   LICEO COMERCIAL JERARDO MUNOZ CAMPOS LICEO COMERCIAL JERARDO MUNOZ CAMPO
304      LICEO BICENTENARIO ANDRES SABELLA                LICEO ANDRES SABELLA
305    ESCUELA HUMBERTO GONZALEZ ECHEGOYEN          ESCUELA GONZALEZ ECHEGOYEN
341          COLEGIO PARTICULAR ADVENTISTA       ESCUELA PARTICULAR ADVENTISTA
686 ESCUELA BÁSICA MARIO AQUILES RODRIGUEZ     ESCUELA MARIO AQUILES RODRIGUEZ

--- Discrepancias 2018/media (muestra) ---
 rbd                          nom_rbd_ive                             nom_rbd
 280 LICEO COMERCIAL JERARDO MUNOZ CAMPOS LICEO COMERCIAL JERARDO MUNOZ CAMPO
 283            LICEO TECNICO ANTOFAGASTA        LICEO TECNICO DE ANTOFAGASTA
 304    LICEO BICENTENARIO ANDRES SABELLA                LICEO ANDRES SABELLA
 341        COLEGIO PARTICULAR ADVENTISTA       ESCUELA PARTICULAR ADVENTISTA
1316              COLEGIO MARIE POUSSEPIN      

In [5]:
# ==============================================================================
# CELDA 5: CONSOLIDACIÓN IVE BIENIO 2018-19 (básica + media combinadas)
# ==============================================================================
def combinar_niveles(ive_basica_dict, ive_media_dict, anios):
    pool_basica = pd.concat([ive_basica_dict[a][['rbd', 'ive_basica']] for a in anios], ignore_index=True)
    pool_media = pd.concat([ive_media_dict[a][['rbd', 'ive_media']] for a in anios], ignore_index=True)

    basica_avg = pool_basica.groupby('rbd', as_index=False).mean()
    media_avg = pool_media.groupby('rbd', as_index=False).mean()

    return pd.merge(basica_avg, media_avg, on='rbd', how='outer')

ive_1819 = combinar_niveles(ive_basica, ive_media, [2018, 2019])

# IVE consolidado: promedio de básica y media donde ambos existan (colegios con ambos niveles)
ive_1819['ive_consolidado'] = ive_1819[['ive_basica', 'ive_media']].mean(axis=1)

print(f"Colegios con IVE (2018-19): {len(ive_1819)}")
print(ive_1819.describe())

RUTA_SALIDA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
ive_1819.to_parquet(RUTA_SALIDA + 'ive_2018_19_por_rbd.parquet', index=False)
print("Guardado OK")

Colegios con IVE (2018-19): 8406
        ive_basica    ive_media  ive_consolidado
count  7852.000000  2615.000000      8406.000000
mean      0.883247     0.798628         0.883224
std       0.127913     0.148393         0.124510
min       0.000000     0.000000         0.000000
25%       0.839404     0.715927         0.839608
50%       0.923290     0.837165         0.921518
75%       0.972034     0.913399         0.969241
max       1.000000     1.000000         1.000000
Guardado OK


In [6]:
# ==============================================================================
# CELDA 6: INTEGRAR IVE A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
RUTA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
df_modelo = pd.read_parquet(RUTA + 'tabla_modelo_final_v10.parquet')
ive = pd.read_parquet(RUTA + 'ive_2018_19_por_rbd.parquet')

df_modelo_v11 = pd.merge(df_modelo, ive, on='rbd', how='left', validate='one_to_one')

print(f"Filas: {len(df_modelo_v11)} (antes: {len(df_modelo)})")
print(f"Con dato IVE: {df_modelo_v11['ive_consolidado'].notna().sum()}")

df_modelo_v11.to_parquet(RUTA + 'tabla_modelo_final_v11.parquet', index=False)
print(df_modelo_v11.shape)

Filas: 7754 (antes: 7754)
Con dato IVE: 7752
(7754, 67)


El remanente de establecimientos sin factor Efectividad/Superación clasificable por patrón de nombre corresponde, según las fuentes oficiales del proceso SNED 2026-2027, a categorías fuera del alcance del producto (aulas hospitalarias, contexto de encierro, SENAME), cuya verificación exhaustiva se excluye por no representar al segmento de cliente objetivo (colegios académicos regulares).

In [7]:
# ==============================================================================
# NOTEBOOK NUEVO: FORMATO_LARGO — CELDA 1
# Reconstrucción colegio × ciclo (corrige cluster desalineado + triplica N)
# ==============================================================================
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'

# 1. Features (todas las fuentes integradas, con cluster/indicer/sel que hay que DESCARTAR)
df_v11 = pd.read_parquet(RUTA + 'tabla_modelo_final_v11.parquet')

# 2. SNED por ciclo (sin promediar) — fuente de cluster/indicer/sel correctos por ciclo
df_sned = pd.read_parquet(RUTA + 'sned_maestro_ciclos.parquet')

print("Columnas SNED disponibles:", [c for c in df_sned.columns if c in
      ['RBD','BIENIO_PREMIO','INDICER','SEL','CLUSTER','EFECTIVR','SUPERAR','INICIAR','MEJORAR','INTEGRAR','IGUALDR']])
print("\nCiclos en SNED:", df_sned['BIENIO_PREMIO'].unique())
print("Filas v11:", len(df_v11), "| columnas:", df_v11.shape[1])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Columnas SNED disponibles: ['RBD', 'EFECTIVR', 'SUPERAR', 'INICIAR', 'MEJORAR', 'INTEGRAR', 'IGUALDR', 'CLUSTER', 'INDICER', 'SEL', 'BIENIO_PREMIO']

Ciclos en SNED: ['2016-17' '2018-19' '2020-21' '2022-23' '2024-25']
Filas v11: 7754 | columnas: 67
